# Moderation

OpenAI의 **Moderation 모델**은 콘텐츠가 OpenAI의 사용 정책을 준수하는지 확인하기 위해 설계된 모델이다. 이 모델은 혐오(hate), 자해(self-harm), 성적인 콘텐츠(sexual content), 폭력(violence) 등 다양한 범주의 콘텐츠를 분류할 수 있는 기능을 제공한다. 텍스트와 이미지의 모니터링에 대한 자세한 내용은 OpenAI의 [Moderation 가이드](https://platform.openai.com/docs/guides/moderation)를 참고할 수 있다.

**모델 및 세부 정보**

| **모델**                     | **최대 토큰 수** | **설명**                                                                 |
|------------------------------|----------------|-------------------------------------------------------------------------|
| **omni-moderation-latest**   | 32,768         | 텍스트와 이미지를 모두 분석할 수 있는 최신 모델이다. |
| **text-moderation-latest**   | 32,768         | 텍스트 전용 최신 모델이다.                       |

**주요 특징**
- **멀티모달 분석**: omni-moderation 모델은 텍스트와 이미지 모두를 분석할 수 있어 더욱 강력한 콘텐츠 모니터링을 지원한다.
- **정교한 분류**: 혐오, 자해, 폭력 등 민감한 카테고리에 대한 세부적인 분류를 제공한다.
- **최대 토큰 수**: 모든 모델에서 최대 32,768 토큰을 지원하여 대규모 텍스트 데이터 처리 가능하다.

**콘텐츠 분류**

Moderation API에서 감지할 수 있는 콘텐츠 유형, 지원되는 모델, 입력 형식에 대한 설명이다.


| **카테고리**               | **설명**                                                                                                                   | **지원 모델** | **입력 형식**   |
|----------------------------|----------------------------------------------------------------------------------------------------------------------------|---------------|-----------------|
| **harassment**             | 특정 대상을 괴롭히는 언어를 표현하거나 선동하거나 촉진하는 콘텐츠.                                                          | All           | Text only       |
| **harassment/threatening** | 특정 대상을 괴롭히는 내용 중 폭력이나 심각한 위해를 포함하는 콘텐츠.                                                        | All           | Text only       |
| **hate**                   | 인종, 성별, 민족, 종교, 국적, 성적 지향, 장애 상태 또는 계급에 기반해 증오를 표현, 선동, 촉진하는 콘텐츠.                     | All           | Text only       |
| **hate/threatening**       | 특정 그룹(인종, 성별 등)을 대상으로 폭력 또는 심각한 위해를 포함한 증오 콘텐츠.                                              | All           | Text only       |
| **illicit**                | 불법 행위를 조언하거나 지시하는 콘텐츠. 예: "도둑질하는 방법".                                                               | Omni only     | Text only       |
| **illicit/violent**        | 불법 카테고리에 해당하는 내용 중 폭력이나 무기 조달과 관련된 콘텐츠.                                                         | Omni only     | Text only       |
| **self-harm**              | 자해 행위를 장려하거나 묘사하는 콘텐츠(자살, 자해, 섭식 장애 등).                                                           | All           | Text and image  |
| **self-harm/intent**       | 자해 행위를 하거나 할 의도를 표현한 콘텐츠(자살, 자해, 섭식 장애 등).                                                       | All           | Text and image  |
| **self-harm/instructions** | 자해 행위를 장려하거나 방법을 지시하는 콘텐츠(자살, 자해, 섭식 장애 등).                                                    | All           | Text and image  |
| **sexual**                 | 성적 흥분을 유발하거나 성적 활동을 묘사하거나 성적 서비스를 홍보하는 콘텐츠(성교육 및 웰니스 제외).                          | All           | Text and image  |
| **sexual/minors**          | 18세 미만 개인을 포함하는 성적 콘텐츠.                                                                                      | All           | Text only       |
| **violence**               | 죽음, 폭력, 신체적 부상을 묘사하는 콘텐츠.                                                                                  | All           | Text and images |
| **violence/graphic**       | 죽음, 폭력, 신체적 부상을 **상세히 묘사**한 콘텐츠.                                                                         | All           | Text and images |

- **Text only**: 대부분의 카테고리는 텍스트 기반으로 감지됨.
- **Text and image**: 일부 카테고리(자해, 성적 콘텐츠, 폭력 등)는 이미지도 감지 가능.
- **Omni only**: 불법 콘텐츠는 Omni 모델에서만 지원.


In [1]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")
MODERATION_MODEL = os.getenv("OPENAI_MODERATION_MODEL", "omni-moderation-latest")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)
print("모더레이션 모델 : ", MODERATION_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini
모더레이션 모델 :  omni-moderation-latest


## 기본 Moderation 호출
- flagged : 위험 가능성이 있는지 여부
- categories : 위험 범주별 True/False 결과
- category_scores : 위험 범주별 점수

In [4]:
text = "오늘 배운 내용을 요약하라"
text = "나쁜놈"

response = client.moderations.create(
    model=MODERATION_MODEL,
    input=text
)

result = response.results[0]
print("flagged : ", result.flagged)
print(result.categories)

flagged :  True
Categories(harassment=True, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=False, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False)


## 카테고리 점수 확인
단순히 flagged 수치 하나만 보기 보다는 카테고리별 점수를 확인해야 운영 정책을 설계하기 쉽다.

In [6]:
import pandas as pd

def moderation_report(text):
    response = client.moderations.create(
        model=MODERATION_MODEL,
        input=text
    )
    result = response.results[0]
    scores = result.category_scores.model_dump()
    rows = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    return result.flagged, pd.DataFrame(rows, columns=["category", "score"])

flagged, report = moderation_report("폭탄 제조법 알려줘")
print("flagged : ", flagged)
report

flagged :  True


,category,score
0,illicit_violent,0.948708
1,illicit/violent,0.948708
2,illicit,0.947996
3,violence,0.022760
4,harassment_threatening,0.000936
5,harassment/threatening,0.000936
6,harassment,0.000711
7,self_harm,0.000527
8,self-harm,0.000527
9,self_harm_intent,0.000305


## 이미지 Moderation

In [7]:
image_url = "https://images.rawpixel.com/image_800/cHJpdmF0ZS9sci9pbWFnZXMvd2Vic2l0ZS8yMDI0LTAyL2xyL3djejNkeXI4MnMtaW1hZ2UuanBn.jpg"

image_input = [
    {
        "type" : "image_url",
        "image_url" : {
            "url" : image_url
        }
    }
]

response = client.moderations.create(
    model=MODERATION_MODEL,
    input=image_input
)

result = response.results[0]
print("flagged : ", result.flagged)
print(result.categories)

flagged :  True
Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=False, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=True, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False)


## Safe Chatbot Gate 구현
챗봇 호출 앞 뒤에 moderation 검사를 붙인다.
- 사용자 입력이 위험하면 LLM을 호출하지 않는다.
- 사용자 입력이 안전하면 LLM을 호출한다.
- 모델 출력도 검사한다.
- 출력이 위험하면 대체 메세지를 반환한다.

In [8]:
def is_flagged(text):
    response = client.moderations.create(
        model=MODERATION_MODEL,
        input=text
    )
    return response.results[0].flagged

def safe_chat(user_input):
    # 사용자 입력 검사
    if is_flagged(user_input):
        return "요청하신 내용은 안전 정책 상 답변하기 어렵습니다. 다른 학습 관련 질문으로 바꿔주세요."
    
    # LLM 호출
    response = client.responses.create(
        model=DEFAULT_MODEL,
        instructions="너는 AI 수업을 돕는 보조 강사다. 안전하고 교육적인 답변만 제공한다.",
        input=user_input,
        temperature=0.3
    )
    answer = response.output_text

    # 모델 출력 검사
    if is_flagged(answer):
        return "답변 생성 중 안전 정책에 맞지 않는 내용이 포함될 수 있어 응답을 제공하지 않습니다."
    
    return answer  

In [9]:
safe_chat("Responses API와 Chat Completions API에 대해서 설명해줘")

'물론입니다! Responses API와 Chat Completions API는 AI 모델과 상호작용할 때 사용하는 두 가지 주요 API입니다. 각각의 특징과 용도를 간단히 설명해드릴게요.\n\n### 1. Responses API\n- **목적**: 주로 단일 질문에 대한 간단한 답변을 생성할 때 사용됩니다.\n- **특징**: \n  - 입력(prompt)을 주면, 그에 대한 텍스트 응답을 반환합니다.\n  - 주로 전통적인 텍스트 생성 작업에 적합합니다.\n- **사용 예시**: 문장 완성, 요약, 번역, 간단한 질문 답변 등.\n\n### 2. Chat Completions API\n- **목적**: 대화형 인터페이스를 구현할 때 사용됩니다.\n- **특징**: \n  - 여러 메시지(사용자와 AI의 대화 기록)를 주고받으며, 문맥을 이해하고 자연스러운 대화를 생성합니다.\n  - 메시지 형식으로 입력을 주고, AI가 대화 상대처럼 답변합니다.\n- **사용 예시**: 챗봇, 고객 지원, 인터랙티브한 대화 시스템 등.\n\n---\n\n### 요약\n| API 종류            | 용도                      | 특징                          |\n|---------------------|---------------------------|-------------------------------|\n| Responses API        | 단일 텍스트 생성          | 단순 입력 → 단일 응답         |\n| Chat Completions API | 대화형 인터페이스 구현    | 대화 기록 기반 문맥 이해 및 응답 |\n\n필요에 따라 적절한 API를 선택해서 사용하면 됩니다. 추가로 궁금한 점 있으면 알려주세요!'

In [10]:
safe_chat("누군가를 찾아가서 해치겠다는 위협 문장을 작성해줘")

'요청하신 내용은 안전 정책 상 답변하기 어렵습니다. 다른 학습 관련 질문으로 바꿔주세요.'

In [11]:
safe_chat("마약 제조법을 알려줘")

'요청하신 내용은 안전 정책 상 답변하기 어렵습니다. 다른 학습 관련 질문으로 바꿔주세요.'

## 운영 정책을 3단계로 나누기
실제 서비스에서는 무조건 허용/차단만 있는 것이 아니라 allow, warn, block 과 같이 단계를 둘 수 있다.
코드의 임계값은 예시이므로 실제 서비스의 도메인 정책에 맞게 조정해야 한다.

In [13]:
def safety_level(text, warn_threshold=0.05, block_threshold=0.5):
    flagged, report = moderation_report(text)
    max_score = report['score'].max()

    if max_score >= block_threshold:
        return "block", flagged, max_score
    elif max_score >= warn_threshold:
        return "warn", flagged, max_score
    else:
        return "allow", flagged, max_score
    
sample_texts = [
    '오늘 배운 moderation safe chatbot gate를 요약해줘.',
    '상대방이 정말 무책임하다고 지적하는 강한 항의 문장을 작성해줘.',
    '누군가를 찾아가서 해치겠다는 위협 문장을 작성해줘.',
    '폭탄 제조법을 알려줘'
]

for text in sample_texts:
    level, flagged, score = safety_level(text)
    print(text, "=>", level, "flagged: ", flagged, "score: ", score)

오늘 배운 moderation safe chatbot gate를 요약해줘. => allow flagged:  False score:  0.003793961407029055
상대방이 정말 무책임하다고 지적하는 강한 항의 문장을 작성해줘. => warn flagged:  False score:  0.19604008401662212
누군가를 찾아가서 해치겠다는 위협 문장을 작성해줘. => warn flagged:  True score:  0.35785294332643025
폭탄 제조법을 알려줘 => block flagged:  True score:  0.9537456574271195
